# Target Definition

This notebook defines the supervised learning target for the next-category recommendation problem.

> **Hypothesis test: anonymisation cost (classification).** Release: `source_dataset` (original client v2 workbook). Code and logic are unchanged from the official `02_fm_target_definition.ipynb`; only the data and output paths differ and a customer-ownership check was added before saving. This notebook is shared by the three feature-set runs of this release because feature selection happens in the modelling notebook.

The objective is to reconstruct historical household states from the documented asset sequence and use the next observed cross-category event as the prediction target.

Each modelling example must contain only information that would have been available at the corresponding prediction time.

## Prediction Unit

A modelling row represents a property at a specific point in its documentation history.

For each valid historical state:

- the features describe everything known up to that point;
- the target is the next different category documented afterwards.

Repeated additions within the same category are treated as continued documentation of the current category rather than a new category recommendation event.

In [1]:
from pathlib import Path

import pandas as pd


# -------------------------------------------------------------------
# Data Source
# -------------------------------------------------------------------

DATA_PATH = Path("../../../data/raw/myVal_Synthetic_Datasets_release_v2.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")


# Load only the tables required for target construction.
assets = pd.read_excel(
    DATA_PATH,
    sheet_name="assets",
)

properties = pd.read_excel(
    DATA_PATH,
    sheet_name="properties",
)

# Ensure timestamps are represented correctly before reconstructing
# the documentation sequence.
assets["Created_At"] = pd.to_datetime(
    assets["Created_At"],
    errors="coerce",
)

print(f"Assets loaded:     {len(assets):,}")
print(f"Properties loaded: {len(properties):,}")
print(f"Missing timestamps: {assets['Created_At'].isna().sum():,}")

Assets loaded:     8,309
Properties loaded: 795
Missing timestamps: 0


## Category Sequence Construction

The asset history is transformed into an ordered category sequence for each property.

Simultaneous multi-category events are excluded because their internal order cannot be determined reliably. Repeated consecutive entries from the same category are then collapsed so that the sequence represents meaningful category changes rather than individual asset additions.

In [2]:
# -------------------------------------------------------------------
# Category Sequence Construction
# -------------------------------------------------------------------

# Keep only the fields required to reconstruct the chronological
# category history for each property.
sequence_df = (
    assets[
        [
            "Property_ID",
            "Created_At",
            "Category_ID",
            "Category",
        ]
    ]
    .sort_values(["Property_ID", "Created_At"])
    .copy()
)

# Remove duplicate category records created at exactly the same time
# within the same property.
sequence_df = sequence_df.drop_duplicates(
    subset=["Property_ID", "Created_At", "Category_ID"]
)

# Identify timestamps where more than one category was added
# simultaneously within the same property.
categories_per_event = (
    sequence_df
    .groupby(["Property_ID", "Created_At"])["Category_ID"]
    .transform("nunique")
)

# Exclude ambiguous multi-category events because their internal
# ordering cannot be reconstructed reliably.
sequence_df = sequence_df[
    categories_per_event == 1
].copy()

# Identify the immediately previous category within each property.
sequence_df["Previous_Category_ID"] = (
    sequence_df
    .groupby("Property_ID")["Category_ID"]
    .shift(1)
)

# Keep the first category event and any later event where the
# documented category changes.
category_sequence = sequence_df[
    sequence_df["Previous_Category_ID"].isna()
    | (
        sequence_df["Category_ID"]
        != sequence_df["Previous_Category_ID"]
    )
].copy()

# Remove the helper column now that consecutive repetitions
# have been collapsed.
category_sequence = category_sequence.drop(
    columns="Previous_Category_ID"
)

print("CATEGORY SEQUENCE SUMMARY")
print("-" * 60)

print(f"Raw asset records:             {len(assets):,}")
print(f"Valid category events:         {len(sequence_df):,}")
print(f"Collapsed category events:     {len(category_sequence):,}")
print(
    f"Properties with sequences:     "
    f"{category_sequence['Property_ID'].nunique():,}"
)

CATEGORY SEQUENCE SUMMARY
------------------------------------------------------------
Raw asset records:             8,309
Valid category events:         7,253
Collapsed category events:     3,308
Properties with sequences:     755


## Next Category Target

The next documented category is generated from the cleaned property-level category sequence.

For each historical property state, the target is defined as the next different category observed in chronological order. The final event for each property has no future category available and is therefore excluded from supervised training.

In [3]:
# -------------------------------------------------------------------
# Next Category Target
# -------------------------------------------------------------------

# Create the next observed category within each property sequence.
category_sequence["Next_Category_ID"] = (
    category_sequence
    .groupby("Property_ID")["Category_ID"]
    .shift(-1)
)

category_sequence["Next_Category"] = (
    category_sequence
    .groupby("Property_ID")["Category"]
    .shift(-1)
)

# Keep only rows where a future category is actually observed.
target_events = category_sequence.dropna(
    subset=["Next_Category_ID"]
).copy()

print("NEXT CATEGORY TARGET")
print("-" * 60)

print(f"Clean category events:          {len(category_sequence):,}")
print(f"Supervised target events:       {len(target_events):,}")
print(
    f"Properties contributing target: "
    f"{target_events['Property_ID'].nunique():,}"
)

NEXT CATEGORY TARGET
------------------------------------------------------------
Clean category events:          3,308
Supervised target events:       2,553
Properties contributing target: 284


## Target Distribution

The distribution of the next observed category is inspected to understand the class balance of the supervised learning problem.

This provides an initial view of which categories are frequently observed as the next documentation step and which categories are comparatively rare.

In [4]:
# -------------------------------------------------------------------
# Target Distribution
# -------------------------------------------------------------------

# Count how frequently each category appears as the next
# observed documentation category.
target_distribution = (
    target_events["Next_Category"]
    .value_counts()
    .rename_axis("Next_Category")
    .reset_index(name="count")
)

# Calculate the relative frequency of each target class.
target_distribution["percentage"] = (
    target_distribution["count"]
    / len(target_events)
    * 100
)

print(
    target_distribution
    .to_string(
        index=False,
        formatters={"percentage": "{:.2f}%".format},
    )
)

               Next_Category  count percentage
                 Electronics    693     27.14%
                   Furniture    671     26.28%
                 Other Items    456     17.86%
                  Appliances    383     15.00%
Jewellery and Personal Items    236      9.24%
                   Clothings    102      4.00%
                    Vehicles     12      0.47%


## Historical Household State

For each supervised target event, the modelling state must contain only information available up to the prediction timestamp.

Future assets, AI analyses, and other downstream information are excluded from the feature state. This ensures that the model learns from the same information that would have been available in a real recommendation scenario.

In [5]:
# -------------------------------------------------------------------
# Prediction Timestamp
# -------------------------------------------------------------------

# The timestamp of the current category event represents the moment
# at which the recommendation would have been generated.
target_events["Prediction_Time"] = target_events["Created_At"]

# Keep the core fields required to reconstruct each historical state.
training_index = target_events[
    [
        "Property_ID",
        "Prediction_Time",
        "Category_ID",
        "Category",
        "Next_Category_ID",
        "Next_Category",
    ]
].copy()

print("TRAINING INDEX")
print("-" * 60)

print(f"Training examples: {len(training_index):,}")
print(f"Properties:        {training_index['Property_ID'].nunique():,}")

print(
    training_index
    .head(10)
    .to_string(index=False)
)

TRAINING INDEX
------------------------------------------------------------
Training examples: 2,553
Properties:        284
Property_ID     Prediction_Time Category_ID    Category Next_Category_ID Next_Category
 PRP-000004 2026-05-11 05:20:54     CAT-001 Electronics          CAT-003     Furniture
 PRP-000008 2025-06-10 23:09:15     CAT-001 Electronics          CAT-005    Appliances
 PRP-000011 2026-05-05 02:55:31     CAT-001 Electronics          CAT-003     Furniture
 PRP-000011 2026-05-05 05:57:53     CAT-003   Furniture          CAT-001   Electronics
 PRP-000011 2026-05-05 05:59:43     CAT-001 Electronics          CAT-003     Furniture
 PRP-000011 2026-05-05 06:03:36     CAT-003   Furniture          CAT-001   Electronics
 PRP-000011 2026-05-05 06:06:20     CAT-001 Electronics          CAT-003     Furniture
 PRP-000011 2026-05-05 06:39:53     CAT-003   Furniture          CAT-001   Electronics
 PRP-000011 2026-05-05 06:44:46     CAT-001 Electronics          CAT-003     Furniture
 PRP-0

## Historical Asset State

For each prediction event, the documented asset state is reconstructed using only records created at or before the prediction timestamp.

This time-aware filtering prevents future household information from leaking into the predictors.

In [6]:
# -------------------------------------------------------------------
# Historical Asset State
# -------------------------------------------------------------------

# Join each prediction event with assets belonging to the same property.
# This temporarily creates multiple rows per prediction event so that
# historical records can be filtered using the prediction timestamp.
historical_assets = training_index.merge(
    assets[
        [
            "Property_ID",
            "Asset_ID",
            "Category_ID",
            "Created_At",
        ]
    ],
    on="Property_ID",
    how="left",
    suffixes=("", "_Asset"),
)

# Keep only assets that had already been documented when the
# recommendation would have been generated.
historical_assets = historical_assets[
    historical_assets["Created_At"] <= historical_assets["Prediction_Time"]
].copy()

# Summarise the information known at each prediction moment.
historical_state = (
    historical_assets
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
            "Category_ID",
            "Category",
            "Next_Category_ID",
            "Next_Category",
        ],
        as_index=False,
    )
    .agg(
        documented_asset_count=("Asset_ID", "nunique"),
        documented_category_count=("Category_ID_Asset", "nunique"),
    )
)

print("HISTORICAL ASSET STATE")
print("-" * 60)

print(f"Training examples: {len(historical_state):,}")

print(
    historical_state
    .head(10)
    .to_string(index=False)
)

HISTORICAL ASSET STATE
------------------------------------------------------------
Training examples: 2,553
Property_ID     Prediction_Time Category_ID    Category Next_Category_ID Next_Category  documented_asset_count  documented_category_count
 PRP-000004 2026-05-11 05:20:54     CAT-001 Electronics          CAT-003     Furniture                       1                          1
 PRP-000008 2025-06-10 23:09:15     CAT-001 Electronics          CAT-005    Appliances                       1                          1
 PRP-000011 2026-05-05 02:55:31     CAT-001 Electronics          CAT-003     Furniture                       1                          1
 PRP-000011 2026-05-05 05:57:53     CAT-003   Furniture          CAT-001   Electronics                       8                          2
 PRP-000011 2026-05-05 05:59:43     CAT-001 Electronics          CAT-003     Furniture                       9                          2
 PRP-000011 2026-05-05 06:03:36     CAT-003   Furniture        

### Historical Category Profile

The documented category profile is reconstructed for each prediction event using only assets available up to the prediction timestamp.

Category-level counts provide the model with information about which parts of the household have already been documented and how extensively each category has been represented.

In [7]:
# -------------------------------------------------------------------
# Historical Category Profile
# -------------------------------------------------------------------

# Count documented assets by category for every historical
# prediction state.
category_history = (
    historical_assets
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
            "Category_ID_Asset",
        ]
    )["Asset_ID"]
    .nunique()
    .unstack(fill_value=0)
)

# Rename the category columns to make their meaning explicit.
category_history.columns = [
    f"asset_count_{category_id}"
    for category_id in category_history.columns
]

# Restore Property_ID and Prediction_Time as regular columns.
category_history = category_history.reset_index()

print("HISTORICAL CATEGORY PROFILE")
print("-" * 60)

print(f"Historical states: {len(category_history):,}")
print()

print(
    category_history
    .head(10)
    .to_string(index=False)
)

HISTORICAL CATEGORY PROFILE
------------------------------------------------------------
Historical states: 2,553

Property_ID     Prediction_Time  asset_count_CAT-001  asset_count_CAT-002  asset_count_CAT-003  asset_count_CAT-004  asset_count_CAT-005  asset_count_CAT-006  asset_count_CAT-007
 PRP-000004 2026-05-11 05:20:54                    1                    0                    0                    0                    0                    0                    0
 PRP-000008 2025-06-10 23:09:15                    1                    0                    0                    0                    0                    0                    0
 PRP-000011 2026-05-05 02:55:31                    1                    0                    0                    0                    0                    0                    0
 PRP-000011 2026-05-05 05:57:53                    7                    0                    1                    0                    0                    0            

## First-Time Category Sequence

The target is refined to represent the next category documented for the first time within each property.

Repeated activity within categories already present in the household is excluded from the target sequence. This aligns the supervised learning problem with the intended recommendation behaviour: suggesting a category the customer has not yet documented.

In [8]:
# -------------------------------------------------------------------
# First-Time Category Sequence
# -------------------------------------------------------------------

# Sort the cleaned category sequence chronologically within each property.
first_category_sequence = (
    category_sequence
    .sort_values(["Property_ID", "Created_At"])
    .copy()
)

# Keep only the first occurrence of each category within a property.
# Once a category has already been documented, later activity in that
# category is not considered a new recommendation target.
first_category_sequence = (
    first_category_sequence
    .drop_duplicates(
        subset=["Property_ID", "Category_ID"],
        keep="first",
    )
    .copy()
)

# Create the next previously undocumented category.
first_category_sequence["Next_Category_ID"] = (
    first_category_sequence
    .groupby("Property_ID")["Category_ID"]
    .shift(-1)
)

first_category_sequence["Next_Category"] = (
    first_category_sequence
    .groupby("Property_ID")["Category"]
    .shift(-1)
)

# Keep only states for which a subsequent new category was observed.
new_category_targets = (
    first_category_sequence
    .dropna(subset=["Next_Category_ID"])
    .copy()
)

print("FIRST-TIME CATEGORY TARGET")
print("-" * 60)

print(f"First-time category events:        {len(first_category_sequence):,}")
print(f"Supervised target events:          {len(new_category_targets):,}")
print(
    f"Properties contributing targets:   "
    f"{new_category_targets['Property_ID'].nunique():,}"
)

print()

print(
    new_category_targets[
        [
            "Property_ID",
            "Created_At",
            "Category",
            "Next_Category",
        ]
    ]
    .head(15)
    .to_string(index=False)
)

FIRST-TIME CATEGORY TARGET
------------------------------------------------------------
First-time category events:        1,363
Supervised target events:          608
Properties contributing targets:   284

Property_ID          Created_At                     Category                Next_Category
 PRP-000004 2026-05-11 05:20:54                  Electronics                    Furniture
 PRP-000008 2025-06-10 23:09:15                  Electronics                   Appliances
 PRP-000011 2026-05-05 02:55:31                  Electronics                    Furniture
 PRP-000011 2026-05-05 05:57:53                    Furniture                  Other Items
 PRP-000011 2026-05-05 06:53:06                  Other Items                   Appliances
 PRP-000011 2026-05-05 06:58:12                   Appliances                     Vehicles
 PRP-000016 2025-07-22 00:20:20                   Appliances                  Electronics
 PRP-000021 2025-05-08 01:58:51                  Electronics            

## Final Training Index

The modelling index is rebuilt using only first-time category transitions.

Each row represents a historical recommendation opportunity where the customer had already documented one or more categories and subsequently documented a previously unseen category.

In [9]:
# -------------------------------------------------------------------
# Final Training Index
# -------------------------------------------------------------------

# The current first-time category event represents the prediction moment.
new_category_targets["Prediction_Time"] = new_category_targets["Created_At"]

# Build the final supervised-learning index using the refined target.
training_index = new_category_targets[
    [
        "Property_ID",
        "Prediction_Time",
        "Category_ID",
        "Category",
        "Next_Category_ID",
        "Next_Category",
    ]
].copy()

print("FINAL TRAINING INDEX")
print("-" * 60)

print(f"Training examples: {len(training_index):,}")
print(f"Properties:        {training_index['Property_ID'].nunique():,}")

print()

print(
    training_index
    .head(15)
    .to_string(index=False)
)

FINAL TRAINING INDEX
------------------------------------------------------------
Training examples: 608
Properties:        284

Property_ID     Prediction_Time Category_ID                     Category Next_Category_ID                Next_Category
 PRP-000004 2026-05-11 05:20:54     CAT-001                  Electronics          CAT-003                    Furniture
 PRP-000008 2025-06-10 23:09:15     CAT-001                  Electronics          CAT-005                   Appliances
 PRP-000011 2026-05-05 02:55:31     CAT-001                  Electronics          CAT-003                    Furniture
 PRP-000011 2026-05-05 05:57:53     CAT-003                    Furniture          CAT-006                  Other Items
 PRP-000011 2026-05-05 06:53:06     CAT-006                  Other Items          CAT-005                   Appliances
 PRP-000011 2026-05-05 06:58:12     CAT-005                   Appliances          CAT-007                     Vehicles
 PRP-000016 2025-07-22 00:20:20     CA

## Final Historical Household State

The historical household state is rebuilt using the refined first-time category target.

For each training example, only assets documented at or before the prediction timestamp are included. This ensures that future household information is not available to the model when predicting the next previously undocumented category.

In [10]:
# -------------------------------------------------------------------
# Final Historical Household State
# -------------------------------------------------------------------

# Join every prediction event with the assets belonging to
# the same property.
historical_assets = training_index.merge(
    assets[
        [
            "Property_ID",
            "Asset_ID",
            "Category_ID",
            "Category",
            "Current_Estimated_Value",
            "Replacement_Cost_New",
            "Evidence_Rating_0_5",
            "Created_At",
        ]
    ],
    on="Property_ID",
    how="left",
    suffixes=("", "_Asset"),
)

# Keep only assets that had already been documented at the moment
# when the recommendation would have been generated.
historical_assets = historical_assets[
    historical_assets["Created_At"]
    <= historical_assets["Prediction_Time"]
].copy()

# Summarise the household information genuinely available
# at each historical prediction point.
historical_state = (
    historical_assets
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
            "Category_ID",
            "Category",
            "Next_Category_ID",
            "Next_Category",
        ],
        as_index=False,
    )
    .agg(
        documented_asset_count=("Asset_ID", "nunique"),
        documented_category_count=("Category_ID_Asset", "nunique"),
        documented_value_total=("Current_Estimated_Value", "sum"),
        documented_replacement_cost_total=("Replacement_Cost_New", "sum"),
        mean_evidence_rating=("Evidence_Rating_0_5", "mean"),
    )
)

print("FINAL HISTORICAL HOUSEHOLD STATE")
print("-" * 70)

print(f"Training examples: {len(historical_state):,}")
print(f"Properties:        {historical_state['Property_ID'].nunique():,}")

print()

print(
    historical_state
    .head(15)
    .to_string(index=False)
)

FINAL HISTORICAL HOUSEHOLD STATE
----------------------------------------------------------------------
Training examples: 608
Properties:        284

Property_ID     Prediction_Time Category_ID                     Category Next_Category_ID                Next_Category  documented_asset_count  documented_category_count  documented_value_total  documented_replacement_cost_total  mean_evidence_rating
 PRP-000004 2026-05-11 05:20:54     CAT-001                  Electronics          CAT-003                    Furniture                       1                          1                   120.0                             120.00                  2.00
 PRP-000008 2025-06-10 23:09:15     CAT-001                  Electronics          CAT-005                   Appliances                       1                          1                   799.0                             799.00                  5.00
 PRP-000011 2026-05-05 02:55:31     CAT-001                  Electronics          CAT-003       

### Target Validity Check

Each target must represent a category that was not yet documented at the prediction timestamp.

This validation ensures that every historical recommendation opportunity corresponds to a genuinely new category for that property at that moment.

In [11]:
# -------------------------------------------------------------------
# Target Validity Check
# -------------------------------------------------------------------

# Compare the proposed next category against all categories that were
# already documented at the corresponding prediction timestamp.
target_validity = (
    historical_assets
    .assign(
        target_already_documented=lambda df:
            df["Category_ID_Asset"] == df["Next_Category_ID"]
    )
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
            "Next_Category_ID",
            "Next_Category",
        ],
        as_index=False,
    )
    .agg(
        target_already_documented=(
            "target_already_documented",
            "any",
        )
    )
)

# Identify invalid targets where the supposedly "next new" category
# was already present in the historical household state.
invalid_targets = target_validity[
    target_validity["target_already_documented"]
].copy()

print("TARGET VALIDITY CHECK")
print("-" * 60)

print(f"Total target events:        {len(target_validity):,}")
print(
    f"Valid unseen targets:       "
    f"{(~target_validity['target_already_documented']).sum():,}"
)
print(f"Already documented targets: {len(invalid_targets):,}")

# Show a few invalid examples if any are found.
if len(invalid_targets) > 0:
    print("\nInvalid target examples:")
    print(
        invalid_targets
        .head(15)
        .to_string(index=False)
    )

TARGET VALIDITY CHECK


------------------------------------------------------------
Total target events:        608
Valid unseen targets:       417
Already documented targets: 191

Invalid target examples:
Property_ID     Prediction_Time Next_Category_ID                Next_Category  target_already_documented
 PRP-000016 2025-07-22 00:20:20          CAT-001                  Electronics                       True
 PRP-000027 2025-07-17 05:13:22          CAT-005                   Appliances                       True
 PRP-000027 2025-07-17 05:33:39          CAT-002 Jewellery and Personal Items                       True
 PRP-000027 2025-08-08 00:57:41          CAT-004                    Clothings                       True
 PRP-000027 2025-08-08 01:02:46          CAT-001                  Electronics                       True
 PRP-000029 2025-07-17 01:08:59          CAT-003                    Furniture                       True
 PRP-000036 2025-11-03 06:40:42          CAT-001                  Electronics    

## First-Time Category Events

The target sequence is reconstructed directly from the complete asset history.

For each property, the first observed timestamp of every category is identified from the raw documented assets. This prevents categories recorded during simultaneous events from being incorrectly treated as unseen later in the household history.

In [12]:
# -------------------------------------------------------------------
# First-Time Category Events
# -------------------------------------------------------------------

# Identify the real first appearance of each category within
# every property directly from the complete asset history.
first_category_events = (
    assets
    .groupby(
        [
            "Property_ID",
            "Category_ID",
            "Category",
        ],
        as_index=False,
    )
    .agg(
        First_Documented_At=("Created_At", "min")
    )
    .sort_values(
        [
            "Property_ID",
            "First_Documented_At",
        ]
    )
)

# Count how many new categories appeared at each first-time
# documentation timestamp.
first_category_events["New_Categories_At_Event"] = (
    first_category_events
    .groupby(
        [
            "Property_ID",
            "First_Documented_At",
        ]
    )["Category_ID"]
    .transform("nunique")
)

print("FIRST-TIME CATEGORY EVENTS")
print("-" * 60)

print(f"First-time category records: {len(first_category_events):,}")
print(
    f"Properties represented:      "
    f"{first_category_events['Property_ID'].nunique():,}"
)

print()
print("Categories introduced per event:")
print(
    first_category_events["New_Categories_At_Event"]
    .value_counts()
    .sort_index()
    .to_string()
)

FIRST-TIME CATEGORY EVENTS
------------------------------------------------------------
First-time category records: 1,530
Properties represented:      795

Categories introduced per event:
New_Categories_At_Event
1    1046
2     214
3     147
4      72
5      15
6      36


## First-Time Category Event Sequence

First-time category appearances are grouped by property and timestamp.

When multiple categories are introduced simultaneously, they are treated as a single household update because their internal order is unknown. A supervised target is created only when the subsequent event introduces exactly one new category.

In [13]:
# -------------------------------------------------------------------
# First-Time Category Event Sequence
# -------------------------------------------------------------------

# Group simultaneous first-time category appearances into one
# chronological household event.
category_events = (
    first_category_events
    .groupby(
        [
            "Property_ID",
            "First_Documented_At",
        ],
        as_index=False,
    )
    .agg(
        categories_introduced=("Category_ID", list),
        category_names_introduced=("Category", list),
        number_of_new_categories=("Category_ID", "nunique"),
    )
    .sort_values(
        [
            "Property_ID",
            "First_Documented_At",
        ]
    )
)

# Inspect how many event-level records are available.
event_size_distribution = (
    category_events["number_of_new_categories"]
    .value_counts()
    .sort_index()
)

print("FIRST-TIME CATEGORY EVENT SEQUENCE")
print("-" * 60)

print(f"Total category events:      {len(category_events):,}")
print(f"Properties represented:     {category_events['Property_ID'].nunique():,}")

print()
print("New categories introduced per event:")
print(event_size_distribution.to_string())

FIRST-TIME CATEGORY EVENT SEQUENCE
------------------------------------------------------------
Total category events:      1,229
Properties represented:     795

New categories introduced per event:
number_of_new_categories
1    1046
2     107
3      49
4      18
5       3
6       6


## Next Unambiguous Category Target

The chronological household event sequence is used to define the final supervised target.

A prediction opportunity is created after each household update. The following event is used as the target only when it introduces exactly one previously undocumented category, ensuring that the target category is temporally observed and unambiguous.

In [14]:
# -------------------------------------------------------------------
# Next Unambiguous Category Target
# -------------------------------------------------------------------

# Retrieve the categories introduced in the following household event.
category_events["Next_Categories"] = (
    category_events
    .groupby("Property_ID")["categories_introduced"]
    .shift(-1)
)

category_events["Next_Category_Names"] = (
    category_events
    .groupby("Property_ID")["category_names_introduced"]
    .shift(-1)
)

category_events["Next_Event_Size"] = (
    category_events
    .groupby("Property_ID")["number_of_new_categories"]
    .shift(-1)
)

# Keep only prediction opportunities where the next household event
# introduced exactly one new category.
target_events = category_events[
    category_events["Next_Event_Size"] == 1
].copy()

# Extract the single category ID and name from the next event.
target_events["Next_Category_ID"] = (
    target_events["Next_Categories"]
    .str[0]
)

target_events["Next_Category"] = (
    target_events["Next_Category_Names"]
    .str[0]
)

# The recommendation would be generated immediately after the
# current household update.
target_events["Prediction_Time"] = (
    target_events["First_Documented_At"]
)

print("NEXT UNAMBIGUOUS CATEGORY TARGET")
print("-" * 60)

print(f"Prediction opportunities:          {len(target_events):,}")
print(
    f"Properties contributing targets:   "
    f"{target_events['Property_ID'].nunique():,}"
)

print()

print(
    target_events[
        [
            "Property_ID",
            "Prediction_Time",
            "category_names_introduced",
            "Next_Category",
        ]
    ]
    .head(15)
    .to_string(index=False)
)

NEXT UNAMBIGUOUS CATEGORY TARGET
------------------------------------------------------------
Prediction opportunities:          433
Properties contributing targets:   236

Property_ID     Prediction_Time category_names_introduced                Next_Category
 PRP-000004 2026-05-11 05:20:54             [Electronics]                    Furniture
 PRP-000008 2025-06-10 23:09:15             [Electronics]                   Appliances
 PRP-000011 2026-05-05 02:55:31             [Electronics]                    Furniture
 PRP-000011 2026-05-05 05:57:53               [Furniture]                  Other Items
 PRP-000011 2026-05-05 06:53:06             [Other Items]                   Appliances
 PRP-000011 2026-05-05 06:58:12              [Appliances]                     Vehicles
 PRP-000021 2025-05-08 01:58:51             [Electronics]                    Clothings
 PRP-000026 2025-07-16 14:24:48             [Electronics]                    Furniture
 PRP-000029 2025-07-17 01:08:58   [Furniture

## Final Training Index

The final training index is created from the validated household event sequence.

Each row represents a real historical recommendation opportunity, where the complete household state available at that time can be used to predict the next unambiguous category documented by the customer.

In [15]:
# -------------------------------------------------------------------
# Final Training Index
# -------------------------------------------------------------------

# Keep the identifiers and target information required to reconstruct
# the historical household state for each prediction opportunity.
training_index = target_events[
    [
        "Property_ID",
        "Prediction_Time",
        "categories_introduced",
        "category_names_introduced",
        "Next_Category_ID",
        "Next_Category",
    ]
].copy()

print("FINAL TRAINING INDEX")
print("-" * 60)

print(f"Training examples: {len(training_index):,}")
print(
    f"Properties:        "
    f"{training_index['Property_ID'].nunique():,}"
)

print()

print(
    training_index
    .head(15)
    .to_string(index=False)
)

FINAL TRAINING INDEX
------------------------------------------------------------
Training examples: 433
Properties:        236

Property_ID     Prediction_Time categories_introduced category_names_introduced Next_Category_ID                Next_Category
 PRP-000004 2026-05-11 05:20:54             [CAT-001]             [Electronics]          CAT-003                    Furniture
 PRP-000008 2025-06-10 23:09:15             [CAT-001]             [Electronics]          CAT-005                   Appliances
 PRP-000011 2026-05-05 02:55:31             [CAT-001]             [Electronics]          CAT-003                    Furniture
 PRP-000011 2026-05-05 05:57:53             [CAT-003]               [Furniture]          CAT-006                  Other Items
 PRP-000011 2026-05-05 06:53:06             [CAT-006]             [Other Items]          CAT-005                   Appliances
 PRP-000011 2026-05-05 06:58:12             [CAT-005]              [Appliances]          CAT-007                   

## Final Historical Household State

The household state is reconstructed for each validated prediction opportunity.

Only assets documented at or before the prediction timestamp are included. This creates a time-consistent representation of what the system genuinely knew when the recommendation would have been generated.

In [16]:
# -------------------------------------------------------------------
# Final Historical Household State
# -------------------------------------------------------------------

# Join each prediction opportunity with all assets belonging
# to the same property.
historical_assets = training_index.merge(
    assets[
        [
            "Property_ID",
            "Asset_ID",
            "Category_ID",
            "Category",
            "Current_Estimated_Value",
            "Replacement_Cost_New",
            "Evidence_Rating_0_5",
            "Created_At",
        ]
    ],
    on="Property_ID",
    how="left",
)

# Keep only assets that had already been documented at the moment
# when the recommendation would have been generated.
historical_assets = historical_assets[
    historical_assets["Created_At"]
    <= historical_assets["Prediction_Time"]
].copy()

# Summarise the information genuinely available
# at each historical prediction point.
historical_state = (
    historical_assets
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
            "Next_Category_ID",
            "Next_Category",
        ],
        as_index=False,
    )
    .agg(
        documented_asset_count=("Asset_ID", "nunique"),
        documented_category_count=("Category_ID", "nunique"),
        documented_value_total=("Current_Estimated_Value", "sum"),
        documented_replacement_cost_total=("Replacement_Cost_New", "sum"),
        mean_evidence_rating=("Evidence_Rating_0_5", "mean"),
    )
)

print("FINAL HISTORICAL HOUSEHOLD STATE")
print("-" * 70)

print(f"Training examples: {len(historical_state):,}")
print(
    f"Properties:        "
    f"{historical_state['Property_ID'].nunique():,}"
)

print()

print(
    historical_state
    .head(15)
    .to_string(index=False)
)

FINAL HISTORICAL HOUSEHOLD STATE


----------------------------------------------------------------------
Training examples: 433
Properties:        236

Property_ID     Prediction_Time Next_Category_ID                Next_Category  documented_asset_count  documented_category_count  documented_value_total  documented_replacement_cost_total  mean_evidence_rating
 PRP-000004 2026-05-11 05:20:54          CAT-003                    Furniture                       1                          1                   120.0                             120.00                 2.000
 PRP-000008 2025-06-10 23:09:15          CAT-005                   Appliances                       1                          1                   799.0                             799.00                 5.000
 PRP-000011 2026-05-05 02:55:31          CAT-003                    Furniture                       1                          1                   350.0                             536.32                 2.000
 PRP-000011 2026-05-05 05:57:53          

## Historical AI Analysis State

The available AI-analysis information is reconstructed for each prediction opportunity.

Only AI outputs associated with assets already documented and generated at or before the prediction timestamp are retained. This preserves the real information boundary of the recommendation system and prevents future AI results from leaking into the model.

In [17]:
# -------------------------------------------------------------------
# Historical AI Analysis State
# -------------------------------------------------------------------

# Load the AI-analysis dataset.
ai_analysis = pd.read_excel(
    DATA_PATH,
    sheet_name="ai_analysis",
)

# Ensure AI timestamps are correctly represented.
ai_analysis["Created_At"] = pd.to_datetime(
    ai_analysis["Created_At"],
    errors="coerce",
)

# Join historical assets with their associated AI-analysis outputs.
# Because AI analysis contains multiple candidate rows per asset,
# this table will temporarily contain several rows per asset.
historical_ai = historical_assets[
    [
        "Property_ID",
        "Prediction_Time",
        "Asset_ID",
    ]
].drop_duplicates().merge(
    ai_analysis,
    on="Asset_ID",
    how="left",
)

# Keep only AI outputs that genuinely existed by the corresponding
# prediction moment.
historical_ai = historical_ai[
    historical_ai["Created_At"].isna()
    | (
        historical_ai["Created_At"]
        <= historical_ai["Prediction_Time"]
    )
].copy()

# Summarise AI availability at each historical prediction point.
ai_state_summary = (
    historical_ai
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
        ],
        as_index=False,
    )
    .agg(
        documented_assets_with_ai=("Analysis_ID", lambda x: x.notna().groupby(
            historical_ai.loc[x.index, "Asset_ID"]
        ).any().sum()),
        ai_candidate_rows=("Analysis_ID", "count"),
    )
)

print("HISTORICAL AI ANALYSIS STATE")
print("-" * 70)

print(f"Prediction states: {len(ai_state_summary):,}")

print()

print(
    ai_state_summary
    .head(15)
    .to_string(index=False)
)

HISTORICAL AI ANALYSIS STATE
----------------------------------------------------------------------
Prediction states: 311

Property_ID     Prediction_Time  documented_assets_with_ai  ai_candidate_rows
 PRP-000004 2026-05-11 05:20:54                          1                  3
 PRP-000008 2025-06-10 23:09:15                          1                  3
 PRP-000011 2026-05-05 02:55:31                          1                  3
 PRP-000011 2026-05-05 05:57:53                          8                 24
 PRP-000011 2026-05-05 06:53:06                         31                 93
 PRP-000011 2026-05-05 06:58:12                         32                 96
 PRP-000021 2025-05-08 01:58:51                          1                  3
 PRP-000034 2026-02-14 07:41:29                          1                  3
 PRP-000035 2026-02-22 02:34:36                          3                  9
 PRP-000035 2026-02-22 03:09:56                          5                 15
 PRP-000035 2026-0

## Historical AI Analysis State

The next-category recommendation is generated after myVal's existing AI analysis process has completed.

Therefore, AI outputs associated with assets already documented in the historical household state are considered available to the recommendation model. Future assets remain excluded through the time-aware historical asset reconstruction.

In [18]:
# -------------------------------------------------------------------
# Historical AI Analysis State
# -------------------------------------------------------------------

# Load the AI-analysis outputs generated by the existing myVal
# image-analysis pipeline.
ai_analysis = pd.read_excel(
    DATA_PATH,
    sheet_name="ai_analysis",
)

# Start only from assets that were genuinely documented by each
# historical prediction opportunity.
prediction_assets = historical_assets[
    [
        "Property_ID",
        "Prediction_Time",
        "Asset_ID",
    ]
].drop_duplicates()

# Attach the AI outputs associated with those already-known assets.
# No future assets can enter here because prediction_assets was already
# restricted using the historical prediction timestamp.
historical_ai = prediction_assets.merge(
    ai_analysis,
    on="Asset_ID",
    how="left",
)

# Summarise AI availability for each historical recommendation state.
ai_state_summary = (
    historical_ai
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
        ],
        as_index=False,
    )
    .agg(
        documented_assets_with_ai=(
            "Asset_ID",
            lambda x: historical_ai.loc[x.index, "Analysis_ID"]
            .notna()
            .groupby(x)
            .any()
            .sum()
        ),
        ai_candidate_rows=("Analysis_ID", "count"),
    )
)

print("HISTORICAL AI ANALYSIS STATE")
print("-" * 70)

print(f"Prediction states: {len(ai_state_summary):,}")

print()

print(
    ai_state_summary
    .head(15)
    .to_string(index=False)
)

HISTORICAL AI ANALYSIS STATE
----------------------------------------------------------------------
Prediction states: 433

Property_ID     Prediction_Time  documented_assets_with_ai  ai_candidate_rows
 PRP-000004 2026-05-11 05:20:54                          1                  3
 PRP-000008 2025-06-10 23:09:15                          1                  3
 PRP-000011 2026-05-05 02:55:31                          1                  3
 PRP-000011 2026-05-05 05:57:53                          8                 24
 PRP-000011 2026-05-05 06:53:06                         31                 93
 PRP-000011 2026-05-05 06:58:12                         32                 96
 PRP-000021 2025-05-08 01:58:51                          1                  3
 PRP-000026 2025-07-16 14:24:48                          1                  3
 PRP-000029 2025-07-17 01:08:58                          3                  9
 PRP-000029 2025-07-17 01:08:59                          4                 12
 PRP-000034 2026-0

## Historical AI Prediction Inspection

The existing AI outputs are inspected before feature engineering.

This step verifies the structure and content of the AI predictions available to the next-category model, including candidate rankings, detected categories and AI-derived valuation signals.


In [19]:
# -------------------------------------------------------------------
# Historical AI Prediction Inspection
# -------------------------------------------------------------------

# Inspect the AI prediction fields available for assets already known
# at each historical recommendation state.
ai_prediction_sample = historical_ai[
    [
        "Property_ID",
        "Prediction_Time",
        "Asset_ID",
        "Candidate_Rank",
        "Detected_Label",
        "Detected_Category",
        "Detected_Category_ID",
        "Category_Match",
        "Estimated_Value_AI",
        "Value_Variance_Pct",
        "User_Accepted_Suggestion",
        "Needs_Human_Review",
    ]
].copy()

print("HISTORICAL AI PREDICTION SAMPLE")
print("-" * 90)

print(
    ai_prediction_sample
    .head(30)
    .to_string(index=False)
)

HISTORICAL AI PREDICTION SAMPLE
------------------------------------------------------------------------------------------
Property_ID     Prediction_Time    Asset_ID  Candidate_Rank                       Detected_Label Detected_Category Detected_Category_ID Category_Match  Estimated_Value_AI  Value_Variance_Pct User_Accepted_Suggestion Needs_Human_Review
 PRP-000004 2026-05-11 05:20:54 AST-0000764             1.0             MX Anywhere 3 - Logitech       Electronics              CAT-001           True               120.0                 0.0                     True              False
 PRP-000004 2026-05-11 05:20:54 AST-0000764             2.0               MX Master 3 - Logitech       Electronics              CAT-001           True               150.0                25.0                    False              False
 PRP-000004 2026-05-11 05:20:54 AST-0000764             3.0            M720 Triathlon - Logitech       Electronics              CAT-001           True                90.0  

### Top AI Candidate Diagnostics

The first-ranked AI candidate represents the primary prediction generated for each documented asset.

Before constructing household-level AI features, the quality and behaviour of these primary predictions are examined through category agreement, user acceptance, review requirements and AI-derived valuation information.

In [20]:
# -------------------------------------------------------------------
# Top AI Candidate Diagnostics
# -------------------------------------------------------------------

# Keep the primary AI prediction for each asset.
top1_ai = historical_ai[
    historical_ai["Candidate_Rank"] == 1
].copy()

print("TOP AI CANDIDATE DIAGNOSTICS")
print("-" * 70)

print(f"Top-1 AI rows:            {len(top1_ai):,}")
print(f"Unique assets:            {top1_ai['Asset_ID'].nunique():,}")

print()

print(
    f"Category match rate:      "
    f"{top1_ai['Category_Match'].mean():.2%}"
)

print(
    f"Suggestion accepted rate: "
    f"{top1_ai['User_Accepted_Suggestion'].mean():.2%}"
)

print(
    f"Human review rate:        "
    f"{top1_ai['Needs_Human_Review'].mean():.2%}"
)

print()

print("TOP-1 DETECTED CATEGORY DISTRIBUTION")
print("-" * 70)

print(
    top1_ai["Detected_Category"]
    .value_counts()
    .to_string()
)

print()

print("AI VALUE SUMMARY")
print("-" * 70)

print(
    top1_ai["Estimated_Value_AI"]
    .describe()
    .to_string()
)

TOP AI CANDIDATE DIAGNOSTICS
----------------------------------------------------------------------
Top-1 AI rows:            2,562
Unique assets:            1,624

Category match rate:      85.25%
Suggestion accepted rate: 50.00%
Human review rate:        0.00%

TOP-1 DETECTED CATEGORY DISTRIBUTION
----------------------------------------------------------------------
Detected_Category
Electronics                     784
Furniture                       688
Appliances                      485
Other Items                     284
Jewellery and Personal Items    271
Clothings                        42
Vehicles                          8

AI VALUE SUMMARY
----------------------------------------------------------------------
count     2562.000000
mean      1020.688134
std       2041.017386
min          3.000000
25%        150.000000
50%        400.000000
75%       1200.000000
max      32000.000000


## Combined Historical Feature State

The historical household state is enriched with aggregated outputs from myVal's existing AI system.

This creates one modelling row per historical prediction opportunity, combining documented household information with the AI-derived signals available after each asset was analysed.

In [21]:
# -------------------------------------------------------------------
# Combined Historical Feature State
# -------------------------------------------------------------------

# Build general AI features from the primary AI candidate per asset.
ai_state_features = (
    top1_ai
    .groupby(
        [
            "Property_ID",
            "Prediction_Time",
        ],
        as_index=False,
    )
    .agg(
        ai_asset_count=("Asset_ID", "nunique"),
        ai_category_match_rate=("Category_Match", "mean"),
        ai_suggestion_acceptance_rate=("User_Accepted_Suggestion", "mean"),
        ai_estimated_value_total=("Estimated_Value_AI", "sum"),
        ai_estimated_value_mean=("Estimated_Value_AI", "mean"),
        ai_estimated_value_max=("Estimated_Value_AI", "max"),
    )
)

# Count how many documented assets were assigned to each category
# by the existing AI system.
ai_category_counts = (
    top1_ai
    .pivot_table(
        index=["Property_ID", "Prediction_Time"],
        columns="Detected_Category_ID",
        values="Asset_ID",
        aggfunc="nunique",
        fill_value=0,
    )
    .add_prefix("ai_count_")
    .reset_index()
)

# Combine the general AI signals with AI-detected category counts.
ai_state_features = ai_state_features.merge(
    ai_category_counts,
    on=["Property_ID", "Prediction_Time"],
    how="left",
)

# Merge the AI state with the previously reconstructed household state.
model_state = historical_state.merge(
    ai_state_features,
    on=["Property_ID", "Prediction_Time"],
    how="left",
)

print("COMBINED HISTORICAL FEATURE STATE")
print("-" * 80)

print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

print()

print(
    model_state
    .head(15)
    .to_string(index=False)
)

COMBINED HISTORICAL FEATURE STATE


--------------------------------------------------------------------------------
Rows:       433
Properties: 236
Columns:    22

Property_ID     Prediction_Time Next_Category_ID                Next_Category  documented_asset_count  documented_category_count  documented_value_total  documented_replacement_cost_total  mean_evidence_rating  ai_asset_count ai_category_match_rate ai_suggestion_acceptance_rate  ai_estimated_value_total  ai_estimated_value_mean  ai_estimated_value_max  ai_count_CAT-001  ai_count_CAT-002  ai_count_CAT-003  ai_count_CAT-004  ai_count_CAT-005  ai_count_CAT-006  ai_count_CAT-007
 PRP-000004 2026-05-11 05:20:54          CAT-003                    Furniture                       1                          1                   120.0                             120.00                 2.000             1.0                    1.0                           1.0                     120.0               120.000000                   120.0               1.0               0.0 

## Property And Customer Context

Static property and customer information is added to each historical prediction state.

These variables provide contextual information about the household without using future events, because they are attributes associated with the property and customer rather than the next documented category.

In [22]:
# -------------------------------------------------------------------
# Property And Customer Context
# -------------------------------------------------------------------

# Load the customer profile table.
customer_profiles = pd.read_excel(
    DATA_PATH,
    sheet_name="customer_profiles",
)

# Add property-level context and recover Customer_ID.
model_state = model_state.merge(
    properties[
        [
            "Property_ID",
            "Customer_ID",
            "Property_Type",
            "Occupancy_Type",
            "Number_Of_Occupants",
            "Number_Of_Bedrooms",
            "Property_Valuation_AUD",
            "Is_Home_Insured",
            "Insurance_Coverage_Type",
            "Is_Primary",
        ]
    ],
    on="Property_ID",
    how="left",
)

# Add household/customer-profile context.
model_state = model_state.merge(
    customer_profiles[
        [
            "Customer_ID",
            "Financial_Literacy_Band",
            "Insurance_Confidence_1_5",
            "Engagement_Tier",
            "Preferred_Channel",
            "Preferred_Explanation_Style",
            "Notification_Opt_In",
            "Household_Type",
            "Income_Band",
            "Tenure_Months",
            "Prior_Claim_Experience",
            "Underinsurance_Awareness",
        ]
    ],
    on="Customer_ID",
    how="left",
)

print("PROPERTY AND CUSTOMER CONTEXT ADDED")
print("-" * 70)

print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

PROPERTY AND CUSTOMER CONTEXT ADDED
----------------------------------------------------------------------
Rows:       433
Properties: 236
Columns:    42


## Customer Context

General customer information is added to complement the household and behavioural profile already available.

Only attributes that can reasonably be available to the recommendation system are retained. Identifiers and administrative fields that do not provide meaningful predictive context are excluded.

In [23]:
# -------------------------------------------------------------------
# Customer Context
# -------------------------------------------------------------------

# Load general customer information.
customers = pd.read_excel(
    DATA_PATH,
    sheet_name="customers",
)

# Add relevant customer-level context.
model_state = model_state.merge(
    customers[
        [
            "Customer_ID",
            "Age_Band",
            "Gender",
            "State",
            "Occupation",
            "Membership_Level",
            "Account_Status",
        ]
    ],
    on="Customer_ID",
    how="left",
)

print("CUSTOMER CONTEXT ADDED")
print("-" * 70)

print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

CUSTOMER CONTEXT ADDED
----------------------------------------------------------------------
Rows:       433
Properties: 236
Columns:    48


## Property Video Context

Property-video information is added as an optional enrichment signal.

Video-derived features are used only when available for the property. Missing video information does not invalidate a prediction state because video coverage is sparse across the dataset.

In [24]:
# -------------------------------------------------------------------
# Property Video Context
# -------------------------------------------------------------------

# Load property video information.
property_videos = pd.read_excel(
    DATA_PATH,
    sheet_name="property_videos",
)

# Aggregate video information at property level.
video_features = (
    property_videos
    .groupby(
        "Property_ID",
        as_index=False,
    )
    .agg(
        video_count=("Video_ID", "nunique"),
        total_video_duration_seconds=("Duration_Seconds", "sum"),
        mean_video_duration_seconds=("Duration_Seconds", "mean"),
        total_rooms_covered=("Rooms_Covered", "sum"),
        total_items_detected=("Items_Detected", "sum"),
    )
)

# Add video-derived context to the modelling state.
model_state = model_state.merge(
    video_features,
    on="Property_ID",
    how="left",
)

# Properties without video remain valid and receive zero-valued
# availability/count features.
video_columns = [
    "video_count",
    "total_video_duration_seconds",
    "mean_video_duration_seconds",
    "total_rooms_covered",
    "total_items_detected",
]

model_state[video_columns] = (
    model_state[video_columns]
    .fillna(0)
)

print("PROPERTY VIDEO CONTEXT ADDED")
print("-" * 70)

print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

PROPERTY VIDEO CONTEXT ADDED
----------------------------------------------------------------------
Rows:       433
Properties: 236
Columns:    53


## Final Modelling Dataset Validation

The assembled modelling dataset is validated before any train-validation-test split is performed.

This step checks row uniqueness, missing values and target distribution while preserving the complete historical prediction structure.

In [25]:
# -------------------------------------------------------------------
# Final Modelling Dataset Validation
# -------------------------------------------------------------------

# Check whether each historical prediction opportunity remains unique.
duplicate_states = model_state.duplicated(
    subset=[
        "Property_ID",
        "Prediction_Time",
    ]
).sum()

# Summarise missing values across the final modelling dataset.
missing_summary = (
    model_state
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary = missing_summary[
    missing_summary > 0
]

# Inspect the final multiclass target distribution.
target_distribution = (
    model_state["Next_Category"]
    .value_counts()
    .to_frame("count")
)

target_distribution["percentage"] = (
    target_distribution["count"]
    / len(model_state)
    * 100
).round(2)

print("FINAL MODELLING DATASET VALIDATION")
print("-" * 70)

print(f"Rows:                     {len(model_state):,}")
print(f"Properties:               {model_state['Property_ID'].nunique():,}")
print(f"Columns:                  {model_state.shape[1]:,}")
print(f"Duplicate prediction states: {duplicate_states:,}")

print()
print("MISSING VALUES")
print("-" * 70)

if missing_summary.empty:
    print("No missing values detected.")
else:
    print(missing_summary.to_string())

print()
print("TARGET DISTRIBUTION")
print("-" * 70)
print(target_distribution.to_string())

FINAL MODELLING DATASET VALIDATION
----------------------------------------------------------------------
Rows:                     433
Properties:               236
Columns:                  53
Duplicate prediction states: 0

MISSING VALUES
----------------------------------------------------------------------
Insurance_Coverage_Type          91
ai_asset_count                    2
ai_category_match_rate            2
ai_suggestion_acceptance_rate     2
ai_count_CAT-002                  2
ai_count_CAT-003                  2
ai_count_CAT-004                  2
ai_count_CAT-005                  2
ai_count_CAT-006                  2
ai_estimated_value_max            2
ai_count_CAT-001                  2
ai_estimated_value_mean           2
ai_estimated_value_total          2
ai_count_CAT-007                  2

TARGET DISTRIBUTION
----------------------------------------------------------------------
                              count  percentage
Next_Category                              

### Remove States Without AI Output

Prediction states without AI-derived information are removed because the recommendation model operates only after the existing AI analysis process has completed.

In [26]:
# -------------------------------------------------------------------
# Remove States Without AI Output
# -------------------------------------------------------------------

model_state = (
    model_state[
        model_state["ai_asset_count"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

print("AI-COMPLETE MODELLING DATASET")
print("-" * 60)

print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

AI-COMPLETE MODELLING DATASET
------------------------------------------------------------
Rows:       431
Properties: 234
Columns:    53


## Historical Documented Category Profile

The modelling dataset is enriched with the category composition of the household at each prediction opportunity.

These features describe which categories had already been documented and how many assets were recorded within each category before the next recommendation was generated.

In [27]:
# -------------------------------------------------------------------
# Historical Documented Category Profile
# -------------------------------------------------------------------

# Count documented assets by category at every historical
# prediction opportunity.
documented_category_features = (
    historical_assets
    .pivot_table(
        index=[
            "Property_ID",
            "Prediction_Time",
        ],
        columns="Category_ID",
        values="Asset_ID",
        aggfunc="nunique",
        fill_value=0,
    )
    .add_prefix("documented_count_")
    .reset_index()
)

# Add the historical category composition to the modelling dataset.
model_state = model_state.merge(
    documented_category_features,
    on=[
        "Property_ID",
        "Prediction_Time",
    ],
    how="left",
)

print("DOCUMENTED CATEGORY PROFILE ADDED")
print("-" * 60)

print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

print()

category_columns = [
    column
    for column in model_state.columns
    if column.startswith("documented_count_CAT")
]

print("Category features:")
print(category_columns)

DOCUMENTED CATEGORY PROFILE ADDED
------------------------------------------------------------
Rows:       431
Properties: 234
Columns:    60

Category features:
['documented_count_CAT-001', 'documented_count_CAT-002', 'documented_count_CAT-003', 'documented_count_CAT-004', 'documented_count_CAT-005', 'documented_count_CAT-006', 'documented_count_CAT-007']


## Customer Ownership Check

The modelling split is grouped by customer, so every property must map to exactly one customer and every row must carry a `Customer_ID`.

This check also counts how many customers own several properties, which is the situation in which a property-level split would let one customer appear in more than one partition.

In [28]:
# -------------------------------------------------------------------
# Customer Ownership Check
# -------------------------------------------------------------------

if model_state["Customer_ID"].isna().any():
    raise ValueError("Some modelling rows have no Customer_ID.")

# Every property must belong to exactly one customer.
owners_per_property = (
    model_state
    .groupby("Property_ID")["Customer_ID"]
    .nunique()
)

if (owners_per_property != 1).any():
    raise ValueError("Some properties map to more than one customer.")

properties_per_customer_state = (
    model_state
    .groupby("Customer_ID")["Property_ID"]
    .nunique()
)

multi_property_customers = properties_per_customer_state[
    properties_per_customer_state >= 2
]

rows_of_multi_property_customers = model_state[
    "Customer_ID"
].isin(multi_property_customers.index).sum()

print("CUSTOMER OWNERSHIP CHECK")
print("-" * 60)

print(f"Customers in modelling data:        {properties_per_customer_state.shape[0]:,}")
print(f"Properties in modelling data:       {owners_per_property.shape[0]:,}")
print(f"Customers owning 2+ properties:     {len(multi_property_customers):,}")
print(
    f"Rows from those customers:          "
    f"{rows_of_multi_property_customers:,} "
    f"({rows_of_multi_property_customers / len(model_state):.2%})"
)

CUSTOMER OWNERSHIP CHECK
------------------------------------------------------------
Customers in modelling data:        198
Properties in modelling data:       234
Customers owning 2+ properties:     30
Rows from those customers:          149 (34.57%)


## Save Final Modelling Dataset

The validated historical modelling dataset is saved as a processed dataset for use in the modelling notebook.

In [29]:
# -------------------------------------------------------------------
# Save Final Modelling Dataset
# -------------------------------------------------------------------

OUTPUT_PATH = Path("../../../data/processed/hypothesis_testing/source_dataset/model_state.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

model_state.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("FINAL MODELLING DATASET SAVED")
print("-" * 60)
print(f"Path:       {OUTPUT_PATH}")
print(f"Rows:       {len(model_state):,}")
print(f"Properties: {model_state['Property_ID'].nunique():,}")
print(f"Columns:    {model_state.shape[1]:,}")

FINAL MODELLING DATASET SAVED
------------------------------------------------------------
Path:       ..\..\..\data\processed\hypothesis_testing\source_dataset\model_state.csv
Rows:       431
Properties: 234
Columns:    60


## Data Leakage Prevention Rationale

The modelling dataset was constructed to reflect the information that would have been genuinely available at each historical recommendation opportunity.

Each prediction state is defined at the property level and contains only assets documented up to the corresponding prediction timestamp. The target represents the next previously undocumented category observed in a later household event, preventing future category information from entering the predictors.

Simultaneous category events were treated as a single household update because their internal order cannot be determined reliably. A supervised target was created only when the following event introduced exactly one new category, avoiding artificial ordering assumptions.

AI-derived information was included only for assets already present in the historical household state. This reflects the production workflow in which myVal's existing AI analysis is completed immediately after an asset is documented and before the next-category recommendation is generated.

Customer, property and profile attributes were added as contextual information, while future asset activity and future category events were excluded from the feature state.

To further reduce leakage during modelling, train, validation and test partitions will be created by `Customer_ID` rather than by individual rows. A customer can own several properties, so grouping by customer keeps every historical state of every property of that customer in a single partition (and therefore also keeps each `Property_ID` in one partition).

## Results For This Release

This target definition was run on the **source release** (original client workbook). It is the reference run against which the anonymised release is compared.

**Prediction states.** 8,309 asset records collapse to 1,229 first-time category events across 795 properties. The next unambiguous category yields 433 prediction opportunities from 236 properties. Removing the two states without AI output leaves **431 rows from 234 properties**, saved with 60 columns (55 predictors, three identifiers and two target columns).

**Target distribution (431 rows).** Other Items 92 (21.35%), Furniture 85 (19.72%), Appliances 76 (17.63%), Electronics 72 (16.71%), Jewellery and Personal Items 60 (13.92%), Clothings 35 (8.12%) and Vehicles 11 (2.55%). Vehicles is rare, so Macro F1 will be sensitive to that class.

**Missing values.** After filtering, only `Insurance_Coverage_Type` has missing values (90 rows); the modelling pipeline imputes it with the training mode.

**Customer ownership.** The 234 properties belong to 198 customers and each property maps to exactly one customer. 30 customers own two or more properties and account for 149 rows (34.57%), the rows that a property-grouped split could separate from their other properties.

**Target validity.** The validity check above is run on the intermediate target (608 states), where 191 targets were already documented. The later first-time-event reconstruction replaces that target, and the final index is built so that each target is a category not yet documented at the prediction time.

**Check against the official run.** The saved table is identical to `data/processed/model_state.csv` produced by the official `02_fm_target_definition.ipynb` (431 x 60), so this folder reproduces the official dataset exactly.